# Dimension Module Demo

The `looptools.dimension` module provides symbolic physical dimension handling for control-loop components. It represents units as numerator/denominator lists and supports multiplication, division, and automatic cancellation of common terms.

In [ ]:
from looptools.dimension import Dimension

## Construction

Create dimensions from numerator and denominator unit lists. Use `dimensionless=True` for dimensionless quantities (e.g., ratios, gains).

In [ ]:
# Dimensionless (e.g., controller gains, filter responses)
d_dimless = Dimension(dimensionless=True)
print(f"Dimensionless: {d_dimless.unit_string()!r} -> {repr(d_dimless)}")

# Simple velocity: m/s
d_velocity = Dimension(["m"], ["s"])
print(f"Velocity: {d_velocity.unit_string()!r}")

# Acceleration: m/s²
d_accel = Dimension(["m"], ["s", "s"])
print(f"Acceleration: {d_accel.unit_string()!r}")

# Common terms are reduced on construction
d_reduced = Dimension(["m", "s"], ["s"])
print(f"m*s/s -> {d_reduced.unit_string()!r} (s cancels)")

## Multiplication and Division

Dimensions multiply and divide symbolically, with automatic cancellation between numerators and denominators.

In [ ]:
# m/s * s = m
d1 = Dimension(["m"], ["s"])
d2 = Dimension(["s"], [])
result = d1 * d2
print(f"m/s * s = {result.unit_string()!r}")

# m / s = m/s
d_m = Dimension(["m"], [])
d_s = Dimension(["s"], [])
print(f"m / s = {(d_m / d_s).unit_string()!r}")

# m/s / s = m/s²
print(f"m/s / s = {(d1 / d_s).unit_string()!r}")

# Dimensionless * anything = same dimension
print(f"1 * m/s = {(Dimension(dimensionless=True) * d1).unit_string()!r}")

## Equality and String Representation

Dimensions compare by symbolic equivalence (order-independent). Use `unit_string()` for display and `show()` for bracketed output.

In [ ]:
# Equality is order-independent
da = Dimension(["m", "kg"], ["s"])
db = Dimension(["kg", "m"], ["s"])
print(f"m*kg/s == kg*m/s: {da == db}")

# Multiplicity matters: m² != m
d_m2 = Dimension(["m", "m"], [])
d_m = Dimension(["m"], [])
print(f"m² == m: {d_m2 == d_m}")

# unit_string() and show()
d = Dimension(["V"], ["Hz"])
print(f"unit_string(): {d.unit_string()!r}")
print("show():", end=" "); d.show()

## Use in Components

Components accept a `unit` parameter for dimensional tracking. Most built-in components (PI, LPF, etc.) are dimensionless. Custom components can specify physical units.

In [ ]:
from looptools.component import Component

# Default: dimensionless (e.g., controller output)
c_gain = Component("Gain", sps=80e6, tf=2.0)
print(f"Gain component unit: {c_gain.unit.unit_string() or '(dimensionless)'}")

# Custom component with V/Hz (e.g., mixer, VCO)
c_mixer = Component(
    "Mixer", sps=80e6, tf="1/(1-z^-1)", domain="z",
    unit=Dimension(["V"], ["Hz"])
)
print(f"Mixer component unit: {c_mixer.unit.unit_string()}")